🟢 < Module 6.1 > Data Cleaning & Imputation — Advanced Survey Non-Response Codes & Imputation Strategies

##### 🎯 Context & Objectives

In survey data engineering (such as the European Social Survey), failing to distinguish between **systematic non-response codes** (`66`, `77`, `88`, `99`) and **legitimate extreme scale values** (`0` and `10`) introduces catastrophic bias into machine learning models. Treating a valid response of `0` ("No trust at all") as a missing value deletes extreme opinions and artificially compresses the variance towards the mean.

In this exercise, you will build an automated, auditable data cleaning and imputation pipeline in Python. You will systematically convert ESS missing codes to standard `NaN`s, preserve valid scale boundaries, and compare single-variable/grouped imputation with multivariate algorithms (`IterativeImputer` / MICE).

##### 📑 Exercise Instructions

###### 1. Syntactic / Algorithmic Level (Systematic Missing Code Handling)

Create a Python notebook (`notebooks/06_1_data_cleaning_imputation.ipynb`) that implements the following technical workflow:

- **Explicit Code Mapping (Avoiding Heuristic Pitfalls):**
    
    - Define a explicit dictionary structure mapping specific ESS missing value codes (`66` = Not applicable, `77` = Refusal, `88` = Don't know, `99` = No answer, `999` = Not available) to `np.nan`.
        
    - **Crucial Rule:** Ensure that legitimate scale endpoints (`0` and `10`) for 11-point Likert variables (such as `ppltrst`, `stfdem`, `lrscale`, `happy`, `rlgdgr`) are **strictly preserved** and NEVER converted to `NaN`.
        
- **Missingness Diagnosis (MCAR vs. MAR vs. MNAR):**
    
    - Calculate and display missing data percentages per column across the dataset.
        
    - Filter and inspect rows where non-response occurs to determine whether missingness is completely random or correlated with demographics (e.g., age or education level).
        

###### 2. Applied / Data Level (Grouped vs. Multivariate Imputation)

Apply these transformations to your merged master dataset (`master_country_features` or `surveys`):

- **Grouped Baseline Imputation:**
    
    - Impute missing values in ordinal survey variables (e.g., `trstlgl`, `stfdem`, `polintr`) using country-level or regional-cluster medians (`cntgrp_fc`) using Pandas `.groupby().transform()`.
        
- **Multivariate Imputation (`IterativeImputer` / MICE):**
    
    - Utilize `sklearn.impute.IterativeImputer` (enabling experimental features via `from sklearn.experimental import enable_iterative_imputer`) to model missing features as a function of other respondent answers.
        
- **Distributional Audit & Impact Assessment:**
    
    - Compare the summary statistics (mean, standard deviation, median, IQR) and probability density plots (KDE) of key variables before and after imputation.
        
    - Verify that the proportion of extreme responses (`0`s and `10`s) remains intact after cleaning.
        

##### 🌐 Official References & Documentation

- **European Social Survey Methodology:** [ESS Data Documentation Report — Missing Values Guide](https://www.google.com/search?q=https://www.europeansocialsurvey.org/methodology)
    
- **Scikit-Learn Imputation Guide:** [Imputing Missing Values (`IterativeImputer`)](https://www.google.com/search?q=https://scikit-learn.org/stable/modules/impute.html%23multivariate-feature-imputation)
    
- **Pandas Missing Data Guide:** [Working with Missing Data in Pandas](https://pandas.pydata.org/docs/user_guide/missing_data.html)
    

##### 📚 Recommended Manuals & CheatSheets

- **Scikit-Learn Imputation Cheat Sheet:** [DataCamp Scikit-Learn Missing Value Handling](https://www.google.com/search?q=https://www.datacamp.com/cheat-sheet/scikit-learn-cheat-sheet-python)
    
- **Rubin's Missing Data Theory:** [Little & Rubin — Statistical Analysis with Missing Data Overview](https://www.wiley.com/)
    

💻 **Tu Desafío:** Crea el cuaderno `notebooks/06_1_data_cleaning_imputation.ipynb` en tu entorno local. Implementa el mapeo explícito de códigos (`77`, `88`, `99` $\rightarrow$ `NaN`) respetando los extremos válidos `0` y `10`, aplica `IterativeImputer` y compara las distribuciones resultantes[cite: 4, 6, 7].

# 1. Syntactic / Algorithmic Level (Systematic Missing Code Handling):
# A: Explicit Code Mapping (Avoiding Heuristic Pitfalls): We must to create a mask filter to check unique values count of some different columms, and decide wich of those values
# corresponds to missing value codes from ESS.
# We must follow the data missing protocol from https://www.europeansocialsurvey.org/sites/default/files/2023-06/ESS9_data_protocol_e01_4.pdf
# Also a md, with columm variable collection located in ./DataScienceCourse/data/raw/VariablesESS.md
# Our vectorized mask will be done on the read process of duckdb sql, so we will save memory and will be cleaner. for these we need to prepare the query,
# Predefining the filter dictionaries.
# Our testing variables are in the next cell:

-edulvlb - Highest level of education

Question: What is the highest level of education you have successfully completed?

ESS rounds: ESS5, ESS6, ESS7, ESS8, ESS9, ESS10, ESS10SC, ESS11

Code list
Code comparison
Value	Category
Collapse categories
0	Not completed ISCED level 1
113	ISCED 1, completed primary education
129	Vocational ISCED 2C < 2 years, no access ISCED 3
212	General/pre-vocational ISCED 2A/2B, access ISCED 3 vocational
213	General ISCED 2A, access ISCED 3A general/all 3
221	Vocational ISCED 2C >= 2 years, no access ISCED 3
222	Vocational ISCED 2A/2B, access ISCED 3 vocational
223	Vocational ISCED 2, access ISCED 3 general/all
229	Vocational ISCED 3C < 2 years, no access ISCED 5
311	General ISCED 3 >=2 years, no access ISCED 5
312	General ISCED 3A/3B, access ISCED 5B/lower tier 5A
313	General ISCED 3A, access upper tier ISCED 5A/all 5
321	Vocational ISCED 3C >= 2 years, no access ISCED 5
322	Vocational ISCED 3A, access ISCED 5B/ lower tier 5A
323	Vocational ISCED 3A, access upper tier ISCED 5A/all 5
412	General ISCED 4A/4B, access ISCED 5B/lower tier 5A
413	General ISCED 4A, access upper tier ISCED 5A/all 5
421	ISCED 4 programmes without access ISCED 5
422	Vocational ISCED 4A/4B, access ISCED 5B/lower tier 5A
423	Vocational ISCED 4A, access upper tier ISCED 5A/all 5
510	ISCED 5A short, intermediate/academic/general tertiary below bachelor
520	ISCED 5B short, advanced vocational qualifications
610	ISCED 5A medium, bachelor/equivalent from lower tier tertiary
620	ISCED 5A medium, bachelor/equivalent from upper/single tier tertiary
710	ISCED 5A long, master/equivalent from lower tier tertiary
720	ISCED 5A long, master/equivalent from upper/single tier tertiary
800	ISCED 6, doctoral degree
5555	Other
7777	Refusal*
8888	Don't know*
9999	No answer*
*) Missing Value


-emplno - Number of employees respondent has/had

Question: How many employees (if any) do/did you have?

ESS rounds: ESS1, ESS2, ESS3, ESS4, ESS5, ESS6, ESS7, ESS8, ESS9, ESS10, ESS10SC, ESS11

Code list
Code comparison
Value	Category
66666	Not applicable*
77777	Refusal*
88888	Don't know*
99999	No answer*
*) Missing Value


-emplrel - Employment relation

Question: In your main job are/were you...

ESS rounds: ESS1, ESS2, ESS3, ESS4, ESS5, ESS6, ESS7, ESS8, ESS9, ESS10, ESS10SC, ESS11

Code list
Code comparison
Value	Category
1	Employee
2	Self-employed
3	Working for own family business
6	Not applicable*
7	Refusal*
8	Don't know*
9	No answer*
*) Missing Value

In [2]:
# 1. Define filter dictionary and extract it the results via duckdb queries.
import os
import numpy as np
import pandas as pd
import duckdb as db
from dotenv import load_dotenv

load_dotenv()

# Initialize DuckDB connection
con = db.connect()
con.execute("INSTALL postgres; LOAD postgres;")

attach_query = f"""
ATTACH IF NOT EXISTS 'dbname={os.getenv("PGDATABASE")} 
        user={os.getenv("PGUSER")} 
        password={os.getenv("PGPASSWORD")} 
        host={os.getenv("PGHOST")} 
        port={os.getenv("PGPORT")}' 
AS db (TYPE postgres);
"""
con.execute(attach_query)

# A. Register updated DuckDB filters (preserving code 6 as a structural valid category)
con.execute("""
    -- 1-digit macro: preserves code 6 (Not applicable), maps 7, 8, 9 to NULL
    CREATE OR REPLACE MACRO clean_ess_1d(x) AS 
        CASE 
            WHEN x IN (7, 8, 9) THEN NULL 
            ELSE x 
        END;
        
    -- 4-digit macro for education levels
    CREATE OR REPLACE MACRO clean_ess_4d(x) AS 
        CASE WHEN x IN (7777, 8888, 9999) THEN NULL ELSE x END;
        
    -- 5-digit macro for employee counts
    CREATE OR REPLACE MACRO clean_ess_5d(x) AS 
        CASE WHEN x IN (66666, 77777, 88888, 99999) THEN NULL ELSE x END;
""")

# B. Build and execute the query for our variables
query_custom_3vars = """
CREATE OR REPLACE TABLE surveys_sample_3v AS
SELECT 
    s.cntry,
    s.essround,
    s.emplrel AS emplrel_raw,  -- Keep raw emplrel for tracking routing vs missing
    clean_ess_1d(s.emplrel) AS emplrel,  -- Cleaned 1D (code 6 is preserved, 7-9 mapped to NULL)
    clean_ess_4d(s.edulvlb) AS edulvlb,  -- Cleaned 4D
    clean_ess_5d(s.emplno)  AS emplno    -- Cleaned 5D
FROM db.surveys s
WHERE s.cntry IS NOT NULL;
"""
con.execute(query_custom_3vars)

# C. Extract results to Pandas DataFrame
df_sample_3v = con.sql("SELECT * FROM surveys_sample_3v").df()

# D. Feature Engineering: Create 'EmployType' 
# Value 1: Employee (Wage earner)
# Value 2: Self-employed or family business (Autónomos / Empresarios)
# Value 3: Inactive, unemployed, or non-applicable population (Default)
df_sample_3v['EmployType'] = 3  # Set default value first
df_sample_3v.loc[df_sample_3v['emplrel'] == 1, 'EmployType'] = 1
df_sample_3v.loc[df_sample_3v['emplrel'].isin([2, 3]), 'EmployType'] = 2

# E. Structural assignment for emplno: 
# If the respondent is a wage employee (EmployType == 1), 
# their number of employees is structurally 0, not a missing value to impute.
df_sample_3v.loc[df_sample_3v['EmployType'] == 1, 'emplno'] = df_sample_3v['emplno'].fillna(0)

print(f"✅ ¡Sample dataframe successfully processed with proper routing separation!")
print(f"📊 Final dimensions: {df_sample_3v.shape}")
print("\nDistribution of 'EmployType' (1: Employee, 2: Self-employed, 3: Inactive/Not applicable):")
print(df_sample_3v['EmployType'].value_counts(sort=False))

print("\nFirst lines with clean results and new feature:")
print(df_sample_3v[['cntry', 'emplrel', 'EmployType', 'emplno', 'edulvlb']].head(10))

✅ ¡Sample dataframe successfully processed with proper routing separation!
📊 Final dimensions: (41702, 7)

Distribution of 'EmployType' (1: Employee, 2: Self-employed, 3: Inactive/Not applicable):
EmployType
3     4186
1    32137
2     5379
Name: count, dtype: int64

First lines with clean results and new feature:
  cntry  emplrel  EmployType  emplno  edulvlb
0    AT        6           3    <NA>      322
1    AT        1           1       0      423
2    AT        1           1       0      610
3    AT        1           1       0      422
4    AT        1           1       0      322
5    AT        1           1       0      313
6    AT        1           1       0      212
7    AT        1           1       0      720
8    AT        1           1       0      422
9    AT        1           1       0      212


In [ ]:
# As we can see on the data, the surveys aplies "not applicated, code 66666" to all employed worker.
# Lets make a boolean mask to fill those samples with 0 value.